# Layout Detection — Fine-tune YOLOv8

Обнаружение текстовых блоков на страницах учебников.

**Pipeline:**
1. Подготовка датасета (`ocr/data/prepare.py --mode detection`)
2. Разбивка на train/val
3. Fine-tune YOLOv8m
4. Визуализация результатов
5. Оценка метрик (mAP50, mAP50-95)

**Железо:** RTX 3080ti (12GB VRAM) — batch=16, imgsz=1024

In [ ]:
import os, sys
# Добавляем корень ml/ в путь
sys.path.insert(0, os.path.abspath('../..'))

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

DATA_DIR = Path('../../data/detection')
DATASET_YAML = DATA_DIR / 'dataset.yaml'
WEIGHTS = 'yolov8m.pt'
EPOCHS = 50
BATCH = 16
IMGSZ = 1024
DEVICE = '0'  # GPU 0; поменяйте на 'cpu' если нет GPU
PROJECT = 'runs/detection'
NAME = 'textbook_v1'

print(f'DATA_DIR exists: {DATA_DIR.exists()}')
print(f'Images: {len(list((DATA_DIR / "images").glob("*.png") if (DATA_DIR/"images").exists() else []))}')

## 1. Подготовка данных

Если данные ещё не подготовлены, запустите prepare.py:

In [ ]:
# Запустить только если данных нет
if not DATASET_YAML.exists():
    !python -m ocr.data.prepare \
        --books_dir ../../books \
        --output_dir ../../data/detection \
        --mode detection \
        --pages_per_book 40
else:
    print('Данные уже подготовлены')

## 2. Визуализация примеров датасета

In [ ]:
def show_yolo_annotations(img_path, label_path, ax, title=''):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x0 = (cx - bw/2) * w
                y0 = (cy - bh/2) * h
                rect = patches.Rectangle((x0, y0), bw*w, bh*h,
                                          linewidth=1, edgecolor='red', facecolor='none')
                ax.add_patch(rect)

imgs = sorted((DATA_DIR / 'images').glob('*.png'))[:6] if (DATA_DIR / 'images').exists() else []
if imgs:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, img_p in zip(axes.flat, imgs):
        lbl_p = DATA_DIR / 'labels' / (img_p.stem + '.txt')
        show_yolo_annotations(img_p, lbl_p, ax, title=img_p.stem)
    plt.tight_layout()
    plt.show()
else:
    print('Нет изображений в', DATA_DIR / 'images')

## 3. Train / Val split

In [ ]:
from ocr.detection.train import split_dataset

yaml_path = split_dataset(str(DATA_DIR), val_ratio=0.15)
print(f'Dataset YAML: {yaml_path}')

# Показываем содержимое yaml
with open(yaml_path) as f:
    print(f.read())

## 4. Обучение

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=DEVICE,
    project=PROJECT,
    name=NAME,
    patience=15,
    save=True,
    plots=True,
    val=True,
)

best_weights = Path(PROJECT) / NAME / 'weights' / 'best.pt'
print(f'Лучшие веса: {best_weights}')

## 5. Loss и метрики по эпохам

In [ ]:
import pandas as pd

results_csv = Path(PROJECT) / NAME / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    metrics = [
        ('train/box_loss', 'Train Box Loss'),
        ('train/cls_loss', 'Train Cls Loss'),
        ('val/box_loss', 'Val Box Loss'),
        ('metrics/mAP50(B)', 'mAP@50'),
        ('metrics/mAP50-95(B)', 'mAP@50-95'),
        ('metrics/precision(B)', 'Precision'),
    ]
    
    for ax, (col, title) in zip(axes.flat, metrics):
        if col in df.columns:
            ax.plot(df['epoch'], df[col])
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('Файл results.csv не найден — обучение ещё не завершено')

## 6. Визуализация предсказаний на тестовых страницах

In [ ]:
from ocr.detection.model import LayoutDetector

detector = LayoutDetector(str(best_weights) if best_weights.exists() else None)

val_imgs = sorted((DATA_DIR / 'images' / 'val').glob('*.png'))[:4]
if not val_imgs:
    val_imgs = sorted((DATA_DIR / 'images').glob('*.png'))[:4]

fig, axes = plt.subplots(1, min(4, len(val_imgs)), figsize=(20, 12))
if len(val_imgs) == 1:
    axes = [axes]

for ax, img_path in zip(axes, val_imgs):
    img = Image.open(img_path)
    detections = detector.predict(img, only_text=False)
    
    ax.imshow(img)
    ax.set_title(f'{img_path.stem}\n{len(detections)} регионов', fontsize=9)
    ax.axis('off')
    
    colors = {'text': 'lime', 'title': 'red', 'list': 'cyan', 'figure': 'yellow', 'table': 'magenta'}
    for det in detections:
        x0, y0, x1, y1 = det.bbox
        color = colors.get(det.class_name, 'white')
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0,
                                   linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0-4, f'{det.class_name} {det.conf:.2f}',
                color=color, fontsize=7, backgroundcolor='black')

plt.tight_layout()
plt.show()

## 7. Финальная оценка

In [ ]:
if best_weights.exists():
    model_eval = YOLO(str(best_weights))
    metrics = model_eval.val(data=yaml_path, imgsz=IMGSZ, device=DEVICE)
    print(f'mAP@50:      {metrics.box.map50:.4f}')
    print(f'mAP@50-95:   {metrics.box.map:.4f}')
    print(f'Precision:   {metrics.box.mp:.4f}')
    print(f'Recall:      {metrics.box.mr:.4f}')
else:
    print('Модель ещё не обучена')